In [0]:
from pyspark.sql.functions import col, upper, current_timestamp

df_po_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/raw_data/purchase_orders_raw.csv")
)

In [0]:
df_po_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.pharma_bronze.bronze_purchase_orders")

In [0]:
df_po_silver = (
    df_po_bronze
    .withColumn(
        "po_status",
        upper(col("po_status"))
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    .dropDuplicates(["po_id"])
)

In [0]:
from delta.tables import DeltaTable

delta_target = DeltaTable.forName(
    spark,
    "workspace.pharma_silver.silver_purchase_orders"
)

(
    delta_target.alias("target")
    .merge(
        df_po_silver.alias("source"),
        "target.po_id = source.po_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
df_po_final = spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
)

print("Total Silver PO records:", df_po_final.count())

print("Duplicate PO IDs:")

df_po_final.groupBy("po_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
df_po_bronze_current = spark.table(
    "workspace.pharma_bronze.bronze_purchase_orders"
)

df_po_silver_current = spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
)

print("Bronze records:", df_po_bronze_current.count())
print("Silver records:", df_po_silver_current.count())

In [0]:
null_po_ids = df_po_silver.filter(
    col("po_id").isNull()
).count()

print("NULL PO IDs:", null_po_ids)

In [0]:
duplicate_po_ids = (
    df_po_silver
    .groupBy("po_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate PO IDs:", duplicate_po_ids)

In [0]:
negative_quantity = df_po_silver.filter(
    col("quantity_ordered") < 0
).count()

print("Negative quantities:", negative_quantity)

In [0]:
valid_statuses = [
    "RECEIVED",
    "PENDING",
    "CANCELLED",
    "APPROVED"
]

invalid_status_count = df_po_silver.filter(
    ~col("po_status").isin(valid_statuses)
).count()

print("Invalid PO statuses:", invalid_status_count)